### Dataset Reference:

- https://www.mbta.com/developers/v3-api
- GTFS (General Transit Feed Specification) Data

In [16]:
import pandas as pd
import json

#### Stops Data

In [17]:
stops = pd.read_csv('MBTA_GTFS/stops.txt')
stop_times = pd.read_csv("MBTA_GTFS/stop_times.txt")
trips = pd.read_csv("MBTA_GTFS/trips.txt")
routes = pd.read_csv("MBTA_GTFS/routes.txt")

/var/folders/zv/4j860ckd1c35y4ccg9xkdxcw0000gn/T/ipykernel_65906/1079818808.py:2: DtypeWarning: Columns (0,3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv("MBTA_GTFS/stop_times.txt")
/var/folders/zv/4j860ckd1c35y4ccg9xkdxcw0000gn/T/ipykernel_65906/1079818808.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv("MBTA_GTFS/trips.txt")


In [18]:
trips_routes = trips.merge(routes[['route_id', 'route_type']], on="route_id", how="left")

stop_vehicle_type = stop_times.merge(trips_routes[['trip_id', 'route_type']], on="trip_id", how="left")

stops_with_vehicle_type = stop_vehicle_type.merge(stops, on="stop_id", how="left")

valid_route_types = [0, 1, 2]
subway_stops = stops_with_vehicle_type[stops_with_vehicle_type['route_type'].isin(valid_route_types)]

# Drop duplicates
subway_stops = subway_stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']].drop_duplicates()

# Convert to GeoJSON format
geojson_stops = {
    "type": "FeatureCollection",
    "features": []
}

for _, row in subway_stops.iterrows():
    feature = {
        "type": "Feature",
        "properties": {
            "stop_id": row["stop_id"],
            "stop_name": row["stop_name"]
        },
        "geometry": {
            "type": "Point",
            "coordinates": [row["stop_lon"], row["stop_lat"]]
        }
    }
    geojson_stops["features"].append(feature)

# Save to a GeoJSON file
with open("metro_stations.geojson", "w") as f:
    json.dump(geojson_stops, f, indent=4)

print("Filtered Subway Stops GeoJSON Created: metro_stations.geojson")

        stop_id            stop_name   stop_lat   stop_lon
1638400   70231        Tappan Street  42.338498 -71.138731
1638401   70233            Dean Road  42.337807 -71.141753
1638402   70235     Englewood Avenue  42.336964 -71.145867
1638403   70237     Cleveland Circle  42.336216 -71.149201
1638404   70151              Kenmore  42.348949 -71.095169
1638405   70211  Saint Mary's Street  42.345884 -71.107697
1638406   70213         Hawes Street  42.344758 -71.111761
1638407   70215          Kent Street  42.344117 -71.114097
1638408   70217    Saint Paul Street  42.343340 -71.116927
1638409   70219      Coolidge Corner  42.342274 -71.120915
Total subway stops: 494


#### Lines and Routes

In [4]:
shapes = pd.read_csv("MBTA_GTFS/shapes.txt")

# Filter routes to show only route_type = 1 (Metro/Subway)
routes_metro = routes[routes["route_type"] == 1]

trips_routes = trips.merge(routes, on="route_id", how="inner")

# Convert to GeoJSON
geojson_lines = {
    "type": "FeatureCollection",
    "features": []
}

for shape_id, group in shapes.groupby("shape_id"):
    group = group.sort_values("shape_pt_sequence")

    route_info = trips_routes[trips_routes["shape_id"] == shape_id]

    if route_info.empty:
        continue

    route_color = "#" + route_info["route_color"].values[0] if not route_info.empty else "#000000"
    route_name = route_info["route_short_name"].values[0] if not route_info.empty else "Unknown Route"

    feature = {
        "type": "Feature",
        "properties": {
            "shape_id": shape_id,
            "route_name": route_name,
            "color": route_color
        },
        "geometry": {
            "type": "LineString",
            "coordinates": list(zip(group["shape_pt_lon"], group["shape_pt_lat"]))
        }
    }
    geojson_lines["features"].append(feature)

# Save to a GeoJSON file
with open("metro_lines.geojson", "w") as f:
    json.dump(geojson_lines, f, indent=4)

print("Metro lines GeoJSON created: metro_lines.geojson")


Metro lines GeoJSON created: metro_lines.geojson
